In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd

def extract_pattern_pairs(log_path: str) -> List[Dict[str, Any]]:
    """Parse a summarization log and return pattern name collections per cluster."""
    lines = Path(log_path).read_text(encoding="utf-8").splitlines()

    entries: List[Dict[str, Any]] = []
    cluster_id: int | None = None
    section: str | None = None
    buffer: List[str] = []
    original_names: List[str] = []
    summarized_names: List[str] = []
    thinking: List[str] = []

    def flush(active_section: str | None) -> None:
        nonlocal buffer, original_names, summarized_names, thinking
        if not active_section:
            buffer = []
            return
        raw = "\n".join(buffer).strip()
        buffer = []
        if not raw:
            return
        if active_section == "original":
            try:
                data = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Failed to parse original patterns for cluster {cluster_id}") from exc
            original_names = [item.get("Pattern Name") for item in data if isinstance(item, dict)]
        elif active_section == "summary":
            try:
                data = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Failed to parse summarized patterns for cluster {cluster_id}") from exc
            summarized_names = [
                item.get("Pattern Name")
                for item in data.get("patterns", [])
                if isinstance(item, dict)
            ]
            thinking = data.get("thinkings","")

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("Cluster ") and stripped.split()[1].isdigit():
            flush(section)
            if cluster_id is not None:
                entries.append({
                    "cluster": cluster_id,
                    "original_patterns": original_names,
                    "summarized_patterns": summarized_names,
                })
            cluster_id = int(stripped.split()[1])
            section = None
            original_names = []
            summarized_names = []
        elif stripped == "Original Patterns:":
            flush(section)
            section = "original"
        elif stripped == "Summarized Patterns:":
            flush(section)
            section = "summary"
        elif stripped == "Verification Result:":
            flush(section)
            section = None
        elif section:
            buffer.append(line)

    flush(section)
    if cluster_id is not None:
        entries.append({
            "cluster": cluster_id,
            "original_patterns": original_names,
            "summarized_patterns": summarized_names,
            "thinking": thinking,
        })

    return entries

In [61]:
log_file = "./logs/summarization_log_iter_01.txt"
pattern_pairs_01 = extract_pattern_pairs(log_file)
pattern_pairs_df_01 = pd.DataFrame(pattern_pairs_01)
iter_01_summarized_patterns = set(pattern_pairs_df_01["summarized_patterns"].explode())

{'patterns': [{'Pattern Name': 'LLM Tool Orchestration', 'Problem': 'Large Language Models (LLMs) are inherently limited by their pre-training data, leading to:\n1.  **Knowledge Gaps**: Outdated, general, or domain-specific information, resulting in factual inaccuracies (hallucinations) or inability to answer specific queries.\n2.  **Reasoning Deficiencies**: Weakness in complex numerical, logical, or domain-specific computations.\n3.  **Perceptual Limitations**: Difficulty in understanding and processing diverse, multimodal inputs (e.g., speech, images, foreign languages).\n4.  **Action Constraints**: Inability to perform real-world actions or access dynamic, real-time data.', 'Context': 'AI systems, particularly LLMs, are tasked with complex real-world problems that demand capabilities beyond their internal knowledge or reasoning. These tasks often require high factual accuracy, access to current or specialized data, precise computations, or the ability to interpret varied user input

In [45]:
log_file = "./logs/summarization_log_iter_02.txt"
pattern_pairs_02 = extract_pattern_pairs(log_file)
pattern_pairs_df_02 = pd.DataFrame(pattern_pairs_02)
iter_02_summarized_patterns = set(pattern_pairs_df_02["summarized_patterns"].explode())

In [46]:
log_file = "./logs/summarization_log_iter_03.txt"
pattern_pairs_03 = extract_pattern_pairs(log_file)
pattern_pairs_df_03 = pd.DataFrame(pattern_pairs_03)
iter_03_summarized_patterns = set(pattern_pairs_df_03["summarized_patterns"].explode())

In [59]:
pattern_pairs_01

[{'cluster': 0,
  'original_patterns': ['External Knowledge Augmentation',
   'Domain-Specific Tool Integration',
   'Multimodal Interaction Augmentation',
   'Knowledge Conflict Resolution (in Tool Augmentation)',
   'Tool-Augmented Foundation Model',
   'Tool Augmentation',
   'Retrieval Augmentation',
   'Knowledge Augmentation',
   'LLM-based Tool Learning',
   'Modular Reasoning, Knowledge and Language (MRKL) System'],
  'summarized_patterns': ['LLM Tool Orchestration']},
 {'cluster': 1,
  'original_patterns': ['Tool Use / Tool Augmentation',
   'Task Automation via Tools'],
  'summarized_patterns': ['Tool-Augmented Agent']},
 {'cluster': 2,
  'original_patterns': ['Trust Calibration through Transparency',
   'Inherently Interpretable Models (Interpretability by Design)',
   'LACE (Local Agnostic attribute Contribution Explanation)',
   'Counterfactual Explanations',
   'Transparent Tool-Use Reasoning',
   'xPlain (Interactive Human-in-the-Loop Explanation Framework)'],
  'summari

In [15]:
intercection_patterns = iter_01_summarized_patterns & iter_02_summarized_patterns# & iter_03_summarized_patterns
print(f"Number of patterns common in all 3 iterations: {len(intercection_patterns)}")
print("Common patterns:")
for pattern in intercection_patterns:
    print(pattern)

Number of patterns common in all 3 iterations: 26
Common patterns:
Denoising Pretraining for Foundational Generative Models
Nearest Neighbor Output Augmentation
Augmented Response Synthesis
Exploratory Reasoning (Tree/Graph of Thoughts)
Retrieval-Augmented Generation (RAG)
Autonomous Tool Generation
Progressive Response Disclosure
Reasoning-Action Alignment
LLM Fallback to Inherent Knowledge
Direct LLM Generation
RAG KV Cache Optimization System
Structured Output Generation
Parameter-Efficient LLM Adaptation
Task Conditioning with Control Tokens
Personalized Tool Interaction
AI System Process Transparency and Trust Calibration
User Intent Resolution
LLM-Guided Human Prompt Refinement
Contextual Refinement
Explicit Step-by-Step Reasoning (Chain-of-Thought)
Adversarial Robustness Evaluation
Ambiguity-Robust Demonstrations
In-Prompt Guardrails
Efficient Dense Semantic Retrieval
Proactive Static Planning
Persona and Contextual Framing


In [50]:
pattern_pairs_df_01["original_patterns"] = pattern_pairs_df_01["original_patterns"].apply(lambda x: str(x).replace("[","").replace("]","").replace(",","\n"))
pattern_pairs_df_02["original_patterns"] = pattern_pairs_df_02["original_patterns"].apply(lambda x: str(x).replace("[","").replace("]","").replace(",","\n"))
pattern_pairs_df_03["original_patterns"] = pattern_pairs_df_03["original_patterns"].apply(lambda x: str(x).replace("[","").replace("]","").replace(",","\n"))

In [47]:
pattern_pairs_df_01[["original_patterns","summarized_patterns"]].to_csv("./summarized_patterns/comparison/iter_01_patterns.csv", index=False)
pattern_pairs_df_02[["original_patterns","summarized_patterns"]].to_csv("./summarized_patterns/comparison/iter_02_patterns.csv", index=False)
pattern_pairs_df_03[["original_patterns","summarized_patterns"]].to_csv("./summarized_patterns/comparison/iter_03_patterns.csv", index=False)

In [51]:
pattern_pairs_df_01[["original_patterns","summarized_patterns"]].to_markdown("./summarized_patterns/comparison/iter_01_patterns.md", index=False)
pattern_pairs_df_02[["original_patterns","summarized_patterns"]].to_markdown("./summarized_patterns/comparison/iter_02_patterns.md", index=False)
pattern_pairs_df_03[["original_patterns","summarized_patterns"]].to_markdown("./summarized_patterns/comparison/iter_03_patterns.md", index=False)